In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv('./outputs/Global_Graphite_Projects_final.csv')
print(f"Loaded {len(df)} records across {df['Country (Short Form)'].nunique()} countries")

Loaded 248 records across 33 countries


In [35]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. DATASET OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("1. DATASET OVERVIEW")
print("=" * 70)

print(f"\nTotal records: {len(df)}")
print(f"Countries: {df['Country (Short Form)'].nunique()}")
print(f"Regions: {df['Region'].nunique()}")

print(f"\nBy Status:")
for status, n in df['Status'].value_counts(dropna=False).items():
    print(f"  {status or 'Unknown':25s} {n:4d}  ({100*n/len(df):.0f}%)")

print(f"\nBy Feature Type:")
for ft, n in df['Feature Type'].value_counts(dropna=False).items():
    print(f"  {str(ft):30s} {n:4d}")

cap_pos = df[df['Annual Production Capacity'] > 0]
cap_neg = df[df['Annual Production Capacity'] < 0]
res = df[df['Total Resource (mt)'].notna()]

print(f"\nCapacity data:")
print(f"  Records with capacity > 0:         {len(cap_pos)}")
print(f"  Records with capacity = -999 (unk): {len(cap_neg)}")
print(f"  Total named capacity:              {cap_pos['Annual Production Capacity'].sum():,.0f} mt")

print(f"\nResource data:")
print(f"  Records with resource data:        {len(res)} / {len(df)} ({100*len(res)/len(df):.0f}%)")
print(f"  Total contained graphite:          {res['Contained Graphite (mt)'].sum()/1e6:,.1f} Mt")
print(f"  With M/I/I breakdown:              {df['Measured Resource (mt)'].notna().sum()}")

print(f"\nBy Source Layer:")
for sl, n in df['Source_Layer'].value_counts().items():
    print(f"  {sl:40s} {n:4d}")

1. DATASET OVERVIEW

Total records: 248
Countries: 33
Regions: 9

By Status:
  Exploration                177  (71%)
  Production                  28  (11%)
  Inactive                    16  (6%)
  Feasibility                 13  (5%)
  Development                 11  (4%)
  Care and Maintenance         3  (1%)

By Feature Type:
  nan                             218
  Mines and Quarries               30

Capacity data:
  Records with capacity > 0:         25
  Records with capacity = -999 (unk): 5
  Total named capacity:              907,630 mt

Resource data:
  Records with resource data:        94 / 248 (38%)
  Total contained graphite:          623.6 Mt
  With M/I/I breakdown:              5

By Source Layer:
  USGS_Flake_Graphite_2024                   44
  INDOPAC_Mineral_Exploration                38
  AFR_Mineral_Exploration                    37
  AFR_Mineral_Deposits                       21
  SWA_Mineral_Deposits                       20
  Norway_Gautneb_2020                 

In [36]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. USGS REFERENCE DATA
#
# Production data from multiple MCS editions to enable time-matched comparison.
# Sources:
#   MCS 2015 (pub Jan 2015): 2013-2014 data
#   MCS 2019 (pub Feb 2019): 2017-2018 data
#   MCS 2024 (pub Jan 2024): 2022-2023 data
#   MCS 2026 (pub Feb 2026): 2024-2025 data
# ══════════════════════════════════════════════════════════════════════════════

usgs = pd.DataFrame([
    #                          2014      2018       2023        2025     Reserves
    ('Brazil',               80_000,   95_000,    66_300,    65_000,  74_000_000),
    ('Canada',               30_000,   40_000,     5_470,     8_000,   5_900_000),
    ('China',               780_000,  693_000, 1_210_000, 1_400_000, 100_000_000),
    ('India',               170_000,   35_000,    25_600,    17_000,   8_600_000),
    ('Korea, North',         30_000,    6_000,     8_100,     8_000,   2_000_000),
    ('Madagascar',            5_000,   46_900,    63_000,    80_000,  27_000_000),
    ('Mexico',               22_000,    9_000,     1_300,       740,   3_100_000),
    ('Mozambique',             None,  104_000,    98_000,    60_000,  25_000_000),
    ('Namibia',                None,    3_460,      None,      None,       None),
    ('Norway',                8_000,   16_000,     6_480,     6_600,     600_000),
    ('Pakistan',               None,   14_000,      None,      None,       None),
    ('Russia',               15_000,   25_200,    15_000,    25_000,  14_000_000),
    ('Sri Lanka',             4_000,    4_000,     3_000,     3_200,   1_500_000),
    ('Tanzania',               None,      150,    13_200,    75_000,  18_000_000),
    ('Turkey',               29_000,    2_000,     2_800,     2_200,   6_900_000),
    ('Ukraine',               5_000,   20_000,     1_670,       800,       None),
    ('Vietnam',                None,    5_000,     2_500,       500,   9_700_000),
    ('Zimbabwe',              7_000,    2_000,      None,      None,       None),
    ('Austria',                None,    1_000,       500,       200,       None),
    ('Germany',                None,      800,       180,       140,       None),
    ('Korea, South',           None,     None,     9_620,       500,   1_800_000),
], columns=['Country', 'Prod_2014', 'Prod_2018', 'Prod_2023', 'Prod_2025', 'Reserves_mt'])

usgs_world = {2014: 1_170_000, 2018: 930_000, 2023: 1_580_000, 2025: 1_800_000}
usgs_world_reserves = 310_000_000

print(f"USGS reference: {len(usgs)} producing countries")
print(f"World production: 2014={usgs_world[2014]:,}, 2018={usgs_world[2018]:,}, "
      f"2023={usgs_world[2023]:,}, 2025={usgs_world[2025]:,} mt")
print(f"World reserves: {usgs_world_reserves/1e6:.0f} Mt")

USGS reference: 21 producing countries
World production: 2014=1,170,000, 2018=930,000, 2023=1,580,000, 2025=1,800,000 mt
World reserves: 310 Mt


In [37]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. COUNTRY COVERAGE — Which USGS producers are in our dataset?
# ══════════════════════════════════════════════════════════════════════════════

our_summary = df.groupby('Country (Short Form)').agg(
    total_records=('Name', 'count'),
    production_records=('Status', lambda x: (x == 'Production').sum()),
    total_capacity=('Annual Production Capacity',
                    lambda x: x[x > 0].sum() if (x > 0).any() else 0),
    n_with_capacity=('Annual Production Capacity',
                     lambda x: ((x.notna()) & (x > 0)).sum()),
    has_resources=('Total Resource (mt)', lambda x: x.notna().sum()),
).reset_index().rename(columns={'Country (Short Form)': 'Country'})

coverage = usgs.merge(our_summary, on='Country', how='outer', indicator=True)
coverage['Coverage'] = coverage['_merge'].map({
    'left_only':  '[X] USGS producer - NOT in data',
    'right_only': '[+] In data - NOT a USGS producer',
    'both':       '[v] Covered'
})

print("=" * 100)
print("3. COUNTRY COVERAGE: Our Dataset vs USGS Producing Countries")
print("=" * 100)

for status in ['[v] Covered', '[X] USGS producer - NOT in data', '[+] In data - NOT a USGS producer']:
    subset = coverage[coverage['Coverage'] == status].copy()
    print(f"\n{status} ({len(subset)} countries):")
    print("-" * 90)
    if status == '[v] Covered':
        subset = subset.sort_values('Prod_2025', ascending=False, na_position='last')
        print(subset[['Country', 'Prod_2025', 'total_capacity', 'n_with_capacity',
                       'total_records', 'has_resources']].to_string(index=False))
    elif '[X]' in status:
        print(subset[['Country', 'Prod_2014', 'Prod_2018', 'Prod_2025']].to_string(index=False))
    else:
        print(subset[['Country', 'total_records', 'has_resources', 'total_capacity']].to_string(index=False))

n_usgs = len(usgs)
n_covered = (coverage['Coverage'] == '[v] Covered').sum()
n_missing = (coverage['Coverage'] == '[X] USGS producer - NOT in data').sum()
print(f"\nSummary: {n_covered}/{n_usgs} USGS producing countries covered ({100*n_covered/n_usgs:.0f}%)")
if n_missing > 0:
    missing = coverage[coverage['Coverage'] == '[X] USGS producer - NOT in data']
    print(f"Missing: {', '.join(missing['Country'].tolist())}")

3. COUNTRY COVERAGE: Our Dataset vs USGS Producing Countries

[v] Covered (17 countries):
------------------------------------------------------------------------------------------
     Country  Prod_2025  total_capacity  n_with_capacity  total_records  has_resources
       China  1400000.0        568000.0              7.0           27.0            3.0
  Madagascar    80000.0        169000.0              4.0           27.0            2.0
    Tanzania    75000.0             0.0              0.0           14.0            6.0
      Brazil    65000.0          4000.0              2.0            3.0            1.0
  Mozambique    60000.0          9350.0              2.0            7.0            5.0
      Russia    25000.0             0.0              0.0            1.0            1.0
       India    17000.0         63000.0              2.0           17.0            0.0
      Canada     8000.0             0.0              0.0           11.0           11.0
Korea, North     8000.0          600

In [38]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. PRODUCTION VOLUME COVERAGE
# What % of world production do the countries in our dataset account for?
# ══════════════════════════════════════════════════════════════════════════════

covered = coverage[coverage['Coverage'] == '[v] Covered']
missing = coverage[coverage['Coverage'] == '[X] USGS producer - NOT in data']

print("=" * 70)
print("4. PRODUCTION VOLUME COVERAGE")
print("=" * 70)

for year, col in [(2014, 'Prod_2014'), (2018, 'Prod_2018'),
                  (2023, 'Prod_2023'), (2025, 'Prod_2025')]:
    covered_prod = covered[col].sum()
    missing_prod = missing[col].dropna().sum()
    world = usgs_world[year]
    
    print(f"\n  {year}:")
    print(f"    Countries we cover produced:  {covered_prod:>12,.0f} mt  ({100*covered_prod/world:.1f}% of world)")
    print(f"    Missing countries produced:   {missing_prod:>12,.0f} mt  ({100*missing_prod/world:.1f}%)")
    print(f"    World total (USGS):           {world:>12,.0f} mt")

4. PRODUCTION VOLUME COVERAGE

  2014:
    Countries we cover produced:     1,156,000 mt  (98.8% of world)
    Missing countries produced:         29,000 mt  (2.5%)
    World total (USGS):              1,170,000 mt

  2018:
    Countries we cover produced:     1,104,710 mt  (118.8% of world)
    Missing countries produced:         17,800 mt  (1.9%)
    World total (USGS):                930,000 mt

  2023:
    Countries we cover produced:     1,529,240 mt  (96.8% of world)
    Missing countries produced:          3,480 mt  (0.2%)
    World total (USGS):              1,580,000 mt

  2025:
    Countries we cover produced:     1,750,340 mt  (97.2% of world)
    Missing countries produced:          2,540 mt  (0.1%)
    World total (USGS):              1,800,000 mt


In [39]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. CAPACITY vs PRODUCTION — HEADLINE GAP
# ══════════════════════════════════════════════════════════════════════════════

total_our_capacity = df[df['Annual Production Capacity'] > 0]['Annual Production Capacity'].sum()

print("=" * 70)
print("5. CAPACITY vs PRODUCTION - HEADLINE")
print("=" * 70)

for year in [2014, 2018, 2023, 2025]:
    world = usgs_world[year]
    gap = world - total_our_capacity
    print(f"\n  vs {year} world production:")
    print(f"    Our total named capacity:   {total_our_capacity:>12,.0f} mt")
    print(f"    USGS world production:      {world:>12,.0f} mt")
    print(f"    Gap:                        {gap:>12,.0f} mt ({100*gap/world:.0f}% unaccounted)")
    print(f"    Coverage:                   {100*total_our_capacity/world:>11.0f}%")

5. CAPACITY vs PRODUCTION - HEADLINE

  vs 2014 world production:
    Our total named capacity:        907,630 mt
    USGS world production:         1,170,000 mt
    Gap:                             262,370 mt (22% unaccounted)
    Coverage:                            78%

  vs 2018 world production:
    Our total named capacity:        907,630 mt
    USGS world production:           930,000 mt
    Gap:                              22,370 mt (2% unaccounted)
    Coverage:                            98%

  vs 2023 world production:
    Our total named capacity:        907,630 mt
    USGS world production:         1,580,000 mt
    Gap:                             672,370 mt (43% unaccounted)
    Coverage:                            57%

  vs 2025 world production:
    Our total named capacity:        907,630 mt
    USGS world production:         1,800,000 mt
    Gap:                             892,370 mt (50% unaccounted)
    Coverage:                            50%


In [40]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. CAPACITY vs PRODUCTION — TIME-MATCHED COUNTRY BREAKDOWN
#
# Our capacity data comes from different vintages:
#   2014: LAC records (Brazil, Mexico)
#   2018: Africa, IndoPac, SWA records
#   2023: China MYB records
#   NaN:  Sri Lanka, Vietnam (undated)
#
# This compares each country against BOTH the vintage-year production
# AND the latest 2025 production.
# ══════════════════════════════════════════════════════════════════════════════

cap_by_country = df[df['Annual Production Capacity'] > 0].groupby('Country (Short Form)').agg(
    our_capacity=('Annual Production Capacity', 'sum'),
    n_facilities=('Annual Production Capacity', 'count'),
    ref_year_min=('Reference Year', 'min'),
    ref_year_max=('Reference Year', 'max'),
    ref_year_median=('Reference Year', 'median'),
).reset_index().rename(columns={'Country (Short Form)': 'Country'})

# Snap reference year to nearest USGS data year
available_years = [2014, 2018, 2023, 2025]

def snap_to_usgs_year(ref_year):
    if pd.isna(ref_year):
        return None
    return min(available_years, key=lambda y: abs(y - ref_year))

cap_by_country['data_vintage'] = cap_by_country['ref_year_median'].apply(snap_to_usgs_year)

# Merge with USGS
comparison = cap_by_country.merge(usgs, on='Country', how='left')

# Look up vintage-matched production
def get_vintage_production(row):
    vintage = row['data_vintage']
    if pd.isna(vintage):
        return None
    return row.get(f'Prod_{int(vintage)}')

comparison['prod_at_vintage'] = comparison.apply(get_vintage_production, axis=1)

# Calculate coverage ratios
comparison['cov_vintage_%'] = np.where(
    comparison['prod_at_vintage'] > 0,
    (comparison['our_capacity'] / comparison['prod_at_vintage'] * 100).round(0), None)
comparison['cov_2025_%'] = np.where(
    comparison['Prod_2025'] > 0,
    (comparison['our_capacity'] / comparison['Prod_2025'] * 100).round(0), None)
comparison['gap_vintage'] = comparison['prod_at_vintage'] - comparison['our_capacity']
comparison['gap_2025'] = comparison['Prod_2025'] - comparison['our_capacity']

comparison = comparison.sort_values('Prod_2025', ascending=False, na_position='last')

print("=" * 125)
print("6. CAPACITY vs PRODUCTION - TIME-MATCHED AND CURRENT")
print("=" * 125)
print(f"{'Country':<16} {'Capacity':>10} {'#Fac':>5} {'Vintage':>8} "
      f"{'Prod@Vint':>11} {'Cov@Vint':>9} {'Gap@Vint':>11}  |  "
      f"{'Prod 2025':>11} {'Cov 2025':>9} {'Gap 2025':>11}")
print("-" * 125)

for _, r in comparison.iterrows():
    vintage = int(r['data_vintage']) if pd.notna(r['data_vintage']) else None
    v_str = str(vintage) if vintage else "n/a"
    
    def fmt(val, width=11):
        return f"{val:>{width},.0f}" if pd.notna(val) else f"{'n/a':>{width}}"
    def fmt_pct(val, width=9):
        return f"{val:>{width-1}.0f}%" if pd.notna(val) else f"{'n/a':>{width}}"
    
    print(f"{r['Country']:<16} {r['our_capacity']:>10,.0f} {r['n_facilities']:>5.0f} {v_str:>8} "
          f"{fmt(r['prod_at_vintage'])} {fmt_pct(r['cov_vintage_%'])} {fmt(r['gap_vintage'])}  |  "
          f"{fmt(r['Prod_2025'])} {fmt_pct(r['cov_2025_%'])} {fmt(r['gap_2025'])}")

total_cap = comparison['our_capacity'].sum()
total_pv = comparison['prod_at_vintage'].sum()
total_p25 = comparison['Prod_2025'].sum()
print("-" * 125)
print(f"{'TOTAL':<16} {total_cap:>10,.0f} {comparison['n_facilities'].sum():>5.0f} {'':>8} "
      f"{total_pv:>11,.0f} {100*total_cap/total_pv:>8.0f}% {total_pv-total_cap:>11,.0f}  |  "
      f"{total_p25:>11,.0f} {100*total_cap/total_p25:>8.0f}% {total_p25-total_cap:>11,.0f}")

6. CAPACITY vs PRODUCTION - TIME-MATCHED AND CURRENT
Country            Capacity  #Fac  Vintage   Prod@Vint  Cov@Vint    Gap@Vint  |    Prod 2025  Cov 2025    Gap 2025
-----------------------------------------------------------------------------------------------------------------------------
China               568,000     7     2023   1,210,000       47%     642,000  |    1,400,000       41%     832,000
Madagascar          169,000     4     2018      46,900      360%    -122,100  |       80,000      211%     -89,000
Brazil                4,000     2     2014      80,000        5%      76,000  |       65,000        6%      61,000
Mozambique            9,350     2     2018     104,000        9%      94,650  |       60,000       16%      50,650
India                63,000     2     2018      35,000      180%     -28,000  |       17,000      371%     -46,000
Korea, North          6,000     3     2018       6,000      100%           0  |        8,000       75%       2,000
Sri Lanka       

In [41]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. INTERPRETATION — What changed between vintage year and 2025?
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 95)
print("7. INTERPRETATION - SHIFTS SINCE DATA COLLECTION")
print("=" * 95)

print(f"\n{'Country':<16} {'Vintage':>8} {'Cov@Vintage':>12} {'Cov@2025':>10} "
      f"{'Shift':>10} {'Interpretation'}")
print("-" * 95)

for _, r in comparison.iterrows():
    cv = r['cov_vintage_%']
    c25 = r['cov_2025_%']
    if pd.notna(cv) and pd.notna(c25) and abs(cv - c25) > 20:
        shift = c25 - cv
        vintage = int(r['data_vintage'])
        if shift > 0:
            interp = "Production declined since data collected"
        else:
            interp = "Production grew; data now understates"
        print(f"{r['Country']:<16} {vintage:>8} {cv:>11.0f}% {c25:>9.0f}% "
              f"{shift:>+9.0f}pp  {interp}")

# Zero-capacity producers
print(f"\nUSGS producers with ZERO capacity in our dataset:")
all_usgs_countries = set(usgs['Country'])
our_cap_countries = set(cap_by_country['Country'])
for c in sorted(all_usgs_countries - our_cap_countries):
    p25 = usgs.loc[usgs['Country'] == c, 'Prod_2025'].iloc[0]
    n_rec = len(df[df['Country (Short Form)'] == c])
    if pd.notna(p25) and p25 > 0:
        print(f"  {c}: {p25:,.0f} mt (2025), {n_rec} records (none with capacity)")

7. INTERPRETATION - SHIFTS SINCE DATA COLLECTION

Country           Vintage  Cov@Vintage   Cov@2025      Shift Interpretation
-----------------------------------------------------------------------------------------------
Madagascar           2018         360%       211%      -149pp  Production grew; data now understates
India                2018         180%       371%      +191pp  Production declined since data collected
Korea, North         2018         100%        75%       -25pp  Production grew; data now understates
Mexico               2014         273%      8108%     +7835pp  Production declined since data collected

USGS producers with ZERO capacity in our dataset:
  Austria: 200 mt (2025), 0 records (none with capacity)
  Canada: 8,000 mt (2025), 11 records (none with capacity)
  Germany: 140 mt (2025), 0 records (none with capacity)
  Korea, South: 500 mt (2025), 8 records (none with capacity)
  Norway: 6,600 mt (2025), 28 records (none with capacity)
  Russia: 25,000 mt (20

In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# 8. CAPACITY DATA VINTAGE
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("8. CAPACITY DATA VINTAGE")
print("=" * 90)

cap_records = df[df['Annual Production Capacity'] > 0][
    ['Name', 'Country (Short Form)', 'Annual Production Capacity', 'Reference Year', 'Status']
].copy().sort_values('Reference Year')

print("\n--- Capacity records by data vintage ---")
print(f"{'Ref Year':<12} {'# Records':>10} {'Total Capacity (mt)':>22} {'Countries'}")
print("-" * 90)

year_bins = cap_records.groupby('Reference Year').agg(
    n=('Name', 'count'),
    total_cap=('Annual Production Capacity', 'sum'),
    countries=('Country (Short Form)', lambda x: ', '.join(sorted(x.unique())))
)
for yr, row in year_bins.iterrows():
    yr_str = f"{int(yr)}" if pd.notna(yr) else "Unknown"
    print(f"{yr_str:<12} {row['n']:>10} {row['total_cap']:>22,.0f}   {row['countries']}")

print(f"\n{'TOTAL':<12} {len(cap_records):>10} {cap_records['Annual Production Capacity'].sum():>22,.0f}")

# Country-level vintage assessment
print(f"\n--- Reference year by country (capacity records only) ---")
print(f"{'Country':<20} {'# Fac':>6} {'Capacity (mt)':>14} {'Data Year(s)':<20} {'Assessment'}")
print("-" * 80)

for country, grp in cap_records.groupby('Country (Short Form)'):
    yrs = grp['Reference Year'].dropna()
    cap = grp['Annual Production Capacity'].sum()
    if yrs.empty:
        yr_str, assessment = "Unknown", "Cannot assess"
    elif yrs.max() >= 2023:
        yr_str = f"{int(yrs.min())}-{int(yrs.max())}" if yrs.nunique() > 1 else str(int(yrs.iloc[0]))
        assessment = "Current"
    elif yrs.max() >= 2018:
        yr_str = f"{int(yrs.min())}-{int(yrs.max())}" if yrs.nunique() > 1 else str(int(yrs.iloc[0]))
        assessment = "Moderately dated"
    else:
        yr_str = f"{int(yrs.min())}-{int(yrs.max())}" if yrs.nunique() > 1 else str(int(yrs.iloc[0]))
        assessment = "OUTDATED"
    print(f"{country:<20} {len(grp):>6} {cap:>14,.0f} {yr_str:<20} {assessment}")

8. CAPACITY DATA VINTAGE

--- Capacity records by data vintage ---
Ref Year      # Records    Total Capacity (mt) Countries
------------------------------------------------------------------------------------------
2014                  3                 64,000   Brazil, Mexico
2018                 12                267,350   India, Korea, North, Madagascar, Mozambique, Namibia
2023                  7                568,000   China

TOTAL                25                907,630

--- Reference year by country (capacity records only) ---
Country               # Fac  Capacity (mt) Data Year(s)         Assessment
--------------------------------------------------------------------------------
Brazil                    2          4,000 2014                 OUTDATED
China                     7        568,000 2023                 Current
India                     2         63,000 2018                 Moderately dated
Korea, North              3          6,000 2018                 Moderately 

In [43]:
# ══════════════════════════════════════════════════════════════════════════════
# 9. RESOURCE & RESERVES COVERAGE
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("9. RESOURCE & RESERVES COVERAGE")
print("=" * 80)

res = df[df['Total Resource (mt)'].notna()].copy()

print(f"\nDeposits with resource estimates: {len(res)} / {len(df)} ({100*len(res)/len(df):.0f}%)")
print(f"Countries with resource data:    {res['Country (Short Form)'].nunique()} / {df['Country (Short Form)'].nunique()}")
print(f"Total ore resource:              {res['Total Resource (mt)'].sum()/1e6:,.1f} Mt")
print(f"Total contained graphite:        {res['Contained Graphite (mt)'].sum()/1e6:,.1f} Mt")
print(f"With M/I/I breakdown:            {res['Measured Resource (mt)'].notna().sum()} (Australia only)")

# Country breakdown
print(f"\n{'Country':<20} {'Deposits':>8} {'Ore (Mt)':>10} {'Graphite (Mt)':>14} {'Avg Grade':>10}")
print("-" * 66)

res_by_country = res.groupby('Country (Short Form)').agg(
    deposits=('Name', 'count'),
    ore_Mt=('Total Resource (mt)', lambda x: x.sum()/1e6),
    graphite_Mt=('Contained Graphite (mt)', lambda x: x.sum()/1e6),
    avg_grade=('Resource Grade (TGC%)', 'mean'),
).sort_values('graphite_Mt', ascending=False)

for country, row in res_by_country.iterrows():
    print(f"{country:<20} {row['deposits']:>8} {row['ore_Mt']:>10.1f} "
          f"{row['graphite_Mt']:>14.2f} {row['avg_grade']:>9.1f}%")

print(f"\n{'TOTAL':<20} {res_by_country['deposits'].sum():>8} "
      f"{res_by_country['ore_Mt'].sum():>10.1f} {res_by_country['graphite_Mt'].sum():>14.2f}")

# Compare to USGS reserves
print(f"\n--- Our Resources vs USGS Reserves ---")
print(f"{'Country':<20} {'Our Graphite (Mt)':>18} {'USGS Reserves (Mt)':>20} {'Ratio':>8}")
print("-" * 70)

res_compare = res_by_country.reset_index().merge(
    usgs[['Country', 'Reserves_mt']].dropna(),
    left_on='Country (Short Form)', right_on='Country', how='outer'
)

for _, row in res_compare.sort_values('Reserves_mt', ascending=False, na_position='last').iterrows():
    country = row.get('Country') or row.get('Country (Short Form)', '?')
    our_res = row['graphite_Mt'] if pd.notna(row.get('graphite_Mt')) else None
    usgs_res = row['Reserves_mt'] / 1e6 if pd.notna(row.get('Reserves_mt')) else None
    
    our_str = f"{our_res:>14.1f} Mt" if our_res else f"{'--':>17}"
    usgs_str = f"{usgs_res:>16.1f} Mt" if usgs_res else f"{'--':>19}"
    ratio = f"{our_res/usgs_res:.1f}x" if (our_res and usgs_res and usgs_res > 0) else "--"
    
    print(f"{country:<20} {our_str} {usgs_str} {ratio:>8}")

print(f"\n{'TOTAL':<20} {res_by_country['graphite_Mt'].sum():>14.1f} Mt"
      f" {usgs_world_reserves/1e6:>16.0f} Mt"
      f" {res_by_country['graphite_Mt'].sum()/(usgs_world_reserves/1e6):.1f}x")

print(f"\nNote: 'Resources' (JORC/inferred) are not equivalent to 'Reserves' (USGS, economic).")
print(f"Resources are typically larger. Ratios >1x are expected.")

9. RESOURCE & RESERVES COVERAGE

Deposits with resource estimates: 94 / 248 (38%)
Countries with resource data:    19 / 33
Total ore resource:              6,800.5 Mt
Total contained graphite:        623.6 Mt
With M/I/I breakdown:            5 (Australia only)

Country              Deposits   Ore (Mt)  Graphite (Mt)  Avg Grade
------------------------------------------------------------------
Mozambique                5.0     1937.1         331.70       8.5%
Australia                18.0      975.6          65.84       9.1%
Tanzania                  6.0     1071.6          63.82       8.0%
China                     3.0      730.9          32.62       9.9%
Canada                   11.0      556.8          30.70       7.6%
United States             3.0      571.7          21.70       3.1%
Norway                   26.0      241.7          21.17      11.8%
Russia                    1.0      116.0          15.08      13.0%
Madagascar                2.0      203.2          11.44       5.3%
S

In [44]:
# ══════════════════════════════════════════════════════════════════════════════
# 10. RESOURCE DATA QUALITY
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("10. RESOURCE DATA QUALITY")
print("=" * 70)

print("\n--- Resource data by source ---")
res_by_source = res.groupby('Source_Layer').agg(
    deposits=('Name', 'count'),
    has_grade=('Resource Grade (TGC%)', lambda x: x.notna().sum()),
    has_graphite=('Contained Graphite (mt)', lambda x: x.notna().sum()),
    has_MII=('Measured Resource (mt)', lambda x: x.notna().sum()),
    avg_grade=('Resource Grade (TGC%)', lambda x: x.mean().round(1)),
)
print(res_by_source.to_string())

grades = res['Resource Grade (TGC%)'].dropna()
print(f"\n--- Grade distribution (TGC%) ---")
print(f"  Count:  {len(grades)}")
print(f"  Mean:   {grades.mean():.1f}%")
print(f"  Median: {grades.median():.1f}%")
print(f"  Min:    {grades.min():.1f}%  Max: {grades.max():.1f}%")
print(f"  Std:    {grades.std():.1f}%")

sizes = (res['Contained Graphite (mt)'].dropna() / 1e6)
print(f"\n--- Deposit size distribution (contained graphite, Mt) ---")
print(f"  Count:  {len(sizes)}")
print(f"  Mean:   {sizes.mean():.2f} Mt")
print(f"  Median: {sizes.median():.2f} Mt")
print(f"  <1 Mt:  {(sizes < 1).sum()} ({100*(sizes < 1).sum()/len(sizes):.0f}%)")
print(f"  1-10 Mt: {((sizes >= 1) & (sizes < 10)).sum()}")
print(f"  >10 Mt: {(sizes >= 10).sum()} "
      f"({', '.join(res.loc[sizes[sizes>=10].index, 'Name'].tolist())})")

10. RESOURCE DATA QUALITY

--- Resource data by source ---
                             deposits  has_grade  has_graphite  has_MII  avg_grade
Source_Layer                                                                      
AFR_Mineral_Exploration            12         12            12        0        7.8
AFR_Mineral_Facilities              4          4             4        0        6.5
AUS_Graphite_Deposits              14         14            14        5        9.0
CHN_Mineral_Exploration             1          1             1        0        9.9
CHN_Mineral_Facilities              1          1             1        0       16.0
INDOPAC_Mineral_Exploration         2          2             2        0        5.5
Norway_Gautneb_2020                16         16            16        0       11.7
USGS_Flake_Graphite_2024           44         44            44        0        9.0

--- Grade distribution (TGC%) ---
  Count:  94
  Mean:   9.2%
  Median: 7.7%
  Min:    1.8%  Max: 36.8%
  Std:

In [45]:
# ══════════════════════════════════════════════════════════════════════════════
# 11. DATA FRESHNESS BY REGION
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("11. DATA FRESHNESS BY REGION")
print("=" * 70)

for region in df['Region'].sort_values().unique():
    subset = df[df['Region'] == region]
    ref = subset['Reference Year'].dropna()
    if not ref.empty:
        print(f"  {region:30s}  {len(subset):3d} records  "
              f"years {int(ref.min())}-{int(ref.max())} (median {int(ref.median())})")
    else:
        print(f"  {region:30s}  {len(subset):3d} records  no year data")

11. DATA FRESHNESS BY REGION
  Africa                           70 records  years 2009-2021 (median 2016)
  Australia                        18 records  years 2015-2024 (median 2023)
  China                            27 records  years 2017-2023 (median 2023)
  Europe                           36 records  years 2017-2023 (median 2020)
  Europe/Asia                       1 records  years 2017-2017 (median 2017)
  Indopacific                      45 records  years 2014-2023 (median 2021)
  Latin America & Caribbean         5 records  years 2014-2020 (median 2014)
  North America                    14 records  years 2012-2023 (median 2018)
  SW Asia                          32 records  years 2018-2018 (median 2018)


In [46]:
# ══════════════════════════════════════════════════════════════════════════════
# 12. GAP SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("12. GAP SUMMARY")
print("=" * 70)

# A. Missing producers
missing_prod = coverage[coverage['Coverage'] == '[X] USGS producer - NOT in data']
if not missing_prod.empty:
    print(f"\nA. USGS producing countries NOT in dataset ({len(missing_prod)}):")
    for _, r in missing_prod.iterrows():
        p25 = r['Prod_2025']
        print(f"   {r['Country']}: {p25:,.0f} mt (2025)" if pd.notna(p25)
              else f"   {r['Country']}: production not reported for 2025")

# B. Zero capacity
print(f"\nB. Countries with USGS production but ZERO capacity in our data:")
all_usgs_countries = set(usgs['Country'])
our_cap_countries = set(cap_by_country['Country'])
zero_cap = []
for c in sorted(all_usgs_countries - our_cap_countries):
    p25 = usgs.loc[usgs['Country'] == c, 'Prod_2025'].iloc[0]
    n_rec = len(df[df['Country (Short Form)'] == c])
    if pd.notna(p25) and p25 > 0:
        zero_cap.append((c, p25, n_rec))
zero_cap.sort(key=lambda x: -x[1])
for c, prod, n in zero_cap:
    print(f"   {c}: {prod:,.0f} mt production, {n} records (none with capacity)")

# C. China
china_cap = df.loc[(df['Country (Short Form)'] == 'China') & (df['Annual Production Capacity'] > 0),
                   'Annual Production Capacity'].sum()
china_prod25 = usgs.loc[usgs['Country'] == 'China', 'Prod_2025'].iloc[0]
china_prod23 = usgs.loc[usgs['Country'] == 'China', 'Prod_2023'].iloc[0]
print(f"\nC. China structural gap:")
print(f"   Named capacity:       {china_cap:>12,.0f} mt")
print(f"   2023 production:      {china_prod23:>12,.0f} mt  (coverage: {100*china_cap/china_prod23:.0f}%)")
print(f"   2025 production:      {china_prod25:>12,.0f} mt  (coverage: {100*china_cap/china_prod25:.0f}%)")
print(f"   Untracked (vs 2025):  {china_prod25-china_cap:>12,.0f} mt")

# D. Resource gaps
print(f"\nD. USGS reserve countries with NO deposit-level data:")
our_res_countries = set(res['Country (Short Form)'].unique())
gap_reserves = 0
for _, r in usgs[usgs['Reserves_mt'].notna()].sort_values('Reserves_mt', ascending=False).iterrows():
    if r['Country'] not in our_res_countries:
        gap_reserves += r['Reserves_mt']
        print(f"   {r['Country']}: {r['Reserves_mt']/1e6:.1f} Mt USGS reserves")
print(f"   Total unrepresented: {gap_reserves/1e6:.1f} Mt "
      f"({100*gap_reserves/usgs_world_reserves:.0f}% of world reserves)")

# E. Outdated records
pre_2020 = df[(df['Annual Production Capacity'] > 0) & (df['Reference Year'] < 2020)]
if not pre_2020.empty:
    print(f"\nE. Potentially outdated capacity records ({len(pre_2020)} records, pre-2020):")
    for _, r in pre_2020.sort_values('Annual Production Capacity', ascending=False).iterrows():
        print(f"   {r['Name']} ({r['Country (Short Form)']}): "
              f"{r['Annual Production Capacity']:,.0f} mt, ref year {int(r['Reference Year'])}")

# Overall
print(f"\n{'=' * 70}")
print(f"OVERALL COMPLETENESS")
print(f"{'=' * 70}")
print(f"  USGS producing countries in data:   {n_covered}/{n_usgs} ({100*n_covered/n_usgs:.0f}%)")
print(f"  Named capacity / 2025 production:   {total_our_capacity:,.0f} / {usgs_world[2025]:,} mt "
      f"({100*total_our_capacity/usgs_world[2025]:.0f}%)")
print(f"  Named capacity / vintage production: {total_our_capacity:,.0f} / {total_pv:,.0f} mt "
      f"({100*total_our_capacity/total_pv:.0f}%) [fairer comparison]")
print(f"  Records with capacity:              {len(cap_pos)}/{len(df)} ({100*len(cap_pos)/len(df):.0f}%)")
print(f"  Records with resources:             {len(res)}/{len(df)} ({100*len(res)/len(df):.0f}%)")
print(f"  Resource countries:                 {res['Country (Short Form)'].nunique()}/{df['Country (Short Form)'].nunique()}")

12. GAP SUMMARY

A. USGS producing countries NOT in dataset (4):
   Austria: 200 mt (2025)
   Germany: 140 mt (2025)
   Pakistan: production not reported for 2025
   Turkey: 2,200 mt (2025)

B. Countries with USGS production but ZERO capacity in our data:
   Tanzania: 75,000 mt production, 14 records (none with capacity)
   Russia: 25,000 mt production, 1 records (none with capacity)
   Canada: 8,000 mt production, 11 records (none with capacity)
   Norway: 6,600 mt production, 28 records (none with capacity)
   Turkey: 2,200 mt production, 0 records (none with capacity)
   Ukraine: 800 mt production, 1 records (none with capacity)
   Korea, South: 500 mt production, 8 records (none with capacity)
   Vietnam: 500 mt production, 1 records (none with capacity)
   Austria: 200 mt production, 0 records (none with capacity)
   Germany: 140 mt production, 0 records (none with capacity)

C. China structural gap:
   Named capacity:            568,000 mt
   2023 production:         1,210,000 mt

# VIS